# Explore Scoring Model Behavior

Requirements: `pip install transformers supabase python-dotenv`

You should already ha

In [1]:
!pip install transformers supabase python-dotenv
!pip install --upgrade typing-extensions
!pip install ipywidgets

In [3]:
import os
import ipywidgets as widgets
from IPython.display import HTML

import torch
import numpy as np
import pandas as pd
from transformers import pipeline
from supabase import create_client, Client
from dotenv import load_dotenv
from tqdm.auto import tqdm

from models import ModernBERT_v2, ModelInput

load_dotenv()
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(supabase_url=url, supabase_key=key)

In [4]:
# Get chunk data
chunk_df = pd.read_csv(
    "/home/jovyan/active-projects/shared-resources/strapi-exports/data/strapi_data.csv",
    index_col=0,
)

# There are some duplicate chunks. Likely multiple pages linked to the same chunk.
chunk_df = chunk_df.loc[~chunk_df.index.duplicated(keep="first"), :]
chunk_df.index.value_counts()

chunk_slug
Structure-and-Function-of-the-Respiratory-System-6247       1
Treatment-and-Control-Conditions-4540                       1
Professional-Journals-1503t                                 1
Scholarly-Books-1504t                                       1
Literature-Search-Strategies-1505t                          1
                                                           ..
Reassigning-Immutable-Data-Types-3118                       1
Immutable-Data-Types-Functions-vs-Local-Assignments-3119    1
Printing-Memory-Addresses-3120                              1
Functions-vs-Methods-3121                                   1
Discussion-II-5852                                          1
Name: count, Length: 585, dtype: int64

In [5]:
# Get logs data
client_names = ["hinze_rmp_summer_2025", "middle_georgia"]

logs_response = (
    supabase.table("logs")
    .select("*")
    # .in_("client_name", client_names)  # Optionally filter by client_name
    .eq("api_endpoint", "/score/answer")  # CRI Endpoint
    .order("created_at", desc=True)  # Optionally sort by most recent API calls
    .limit(1_000)  # Number of samples to collect
    .execute()
)

logs_df = pd.json_normalize(logs_response.data)

# Join API query data with content data extracted from Strapi (to get volume slug, page slug, chunk text, question, reference answer)
df = logs_df.merge(
    chunk_df, left_on="request_body.chunk_slug", right_on="chunk_slug", how="left"
)

# Construct DataFrame
columns_to_keep = [
    "volume_slug",
    "request_body.page_slug",
    "request_body.chunk_slug",
    "chunk_text",
    "question",
    "answer",
    "request_body.answer",
    "response_body.level",
    "response_body.score",
    "response_body.feedback",
    "response_body.is_passing",
]

df = df[columns_to_keep]
df = df.rename(
    columns={
        "answer": "reference",
        "request_body.answer": "candidate",
        "chunk_text": "text",
    }
)

# Clean up column names
df.columns = [
    col.replace("request_body.", "").replace("response_body.", "") for col in df.columns
]

# Remove rows without necessary information
df = df[~df["question"].isna() & ~df["text"].isna() & ~df["reference"].isna()]  # Some rows have NaN values

df

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing
0,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Approaches-to-Academic-Discourse-Analysis-5216,Background\nApproaches to Academic Discourse A...,What types of linguistic characteristics have ...,The linguistic characteristics studied in acad...,"hedging devices, verb usage, noun phrase struc...",2,1.886139,"You are on the right track, but your response ...",False
1,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,Using the data to check the consistency of tes...,3,2.364554,Good job. You included some relevant details f...,True
2,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Patterns-of-Variation-Among-University-Registe...,University Registers Along Dimension 3: Situat...,What defines Dimension 3 along university regi...,Dimension 3 defines an absolute polar distinct...,it define the absolute polar distinction betwe...,4,3.210014,Excellent job! Your response demonstrates deep...,True
3,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Data-Coding-5220,Data Coding\n\nAll texts in the corpus were co...,How were the spoken texts transcribed in the c...,Spoken texts were transcribed using a consiste...,it is transcribed using a consistent convension,3,2.787710,Good job. You included some relevant details f...,True
4,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Approaches-to-Academic-Discourse-Analysis-5216,Background\nApproaches to Academic Discourse A...,What types of linguistic characteristics have ...,The linguistic characteristics studied in acad...,the studied on rhetorical organization of clas...,3,2.830444,Good job. You included some relevant details f...,True
...,...,...,...,...,...,...,...,...,...,...,...
995,research-methods-in-psychology,29-correlational-research,Correlation-does-not-Prove-Causation-4649,You have probably heard repeatedly that “Corre...,What are the two reasons mentioned in the text...,"One reason is the directionality problem, whic...",the third variable problem and the directional...,2,1.961525,"You are on the right track, but your response ...",False
996,research-methods-in-psychology,29-correlational-research,Does-Correlational-Research-Always-Involve-Qua...,Does Correlational Research Always Involve Qua...,Explain why correlational research doesn't alw...,The defining feature of correlational research...,correlational research is more about the desig...,2,1.925314,"You are on the right track, but your response ...",False
997,research-methods-in-psychology,28-overview-of-non-experimental-research,Internal-Validity-Revisited-4638,Recall that internal validity is the extent to...,What is internal validity and how does it vary...,Internal validity is the extent to which the d...,the ability to make causal conclusions from yo...,4,3.600016,Excellent job! Your response demonstrates deep...,True
998,research-methods-in-psychology,25-experimentation-and-validity,Construct-Validity-4590,In addition to the generalizability of the res...,How does the number of conditions in an experi...,The number of conditions in an experiment can ...,"more conditions increases construct validity, ...",3,2.356437,Good job. You included some relevant details f...,True


## Score using MPNet

In [6]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lear-lab/short-answer-classification")

# Prepare all inputs at once
inputs = [f"{cand} </s> {ref}" for cand, ref in zip(df['candidate'], df['reference'])]

# Process in batches
results = pipe(inputs, batch_size=8)  # Adjust batch_size based on your GPU memory

# Extract scores
df['mpnet_score'] = [result['score'] if 'score' in result else result['label'] for result in results]

Device set to use cuda:0


## Re-score the data

In [7]:
mb_w_reference_answer_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_multirc_reading_engagement"

MBV2 = ModernBERT_v2(mb_w_reference_answer_path)

Device set to use cuda


In [8]:
def score(df, pipe):
    all_preds = []

    for ex in tqdm(df.to_dict("records")):
        model_input = ModelInput.from_dict(ex)
        pred = pipe(model_input)
        all_preds.append(pred)

    return all_preds


df["new_score"] = score(df, MBV2)

  0%|          | 0/979 [00:00<?, ?it/s]

/home/jovyan/conda_envs/hf/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:140: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


## Look for instances of disagreement between MPNet and MBV

In [9]:
mpnet_threshold = 0.95
new_score_threshold = 3

disagreements = df[(df['mpnet_score'] > mpnet_threshold) & (df['new_score'] < new_score_threshold)][['reference', 'candidate']]
disagreements

,reference,candidate
0,The linguistic characteristics studied in acad...,"hedging devices, verb usage, noun phrase struc..."
27,A monograph is written by a single author or s...,A monograph presents a research topic. It is w...
28,A monograph is written by a single author or s...,A monograph presents a research topic. It is w...
29,A monograph is written by a single author or s...,A monograph presents a research topic. It is w...
35,Educational Testing Service is revising the TO...,check the consistency between the test languag...
...,...,...
935,"Cohen's d is a measure of effect size, which c...",it is a measure of effect size for the differe...
937,The method section should describe all importa...,"a detailed description of the participants, de..."
939,"Title page, Abstract, Introduction, Method, Re...","introduction, method, results, discussion"
986,In a one-group design without a control condit...,"in spontaneous remission, conditions improve o..."


In [10]:
index = 2
print(disagreements.iloc[index]['reference'])
print(disagreements.iloc[index]['candidate'])

A monograph is written by a single author or small group of authors, giving a coherent presentation of a topic, while edited volumes have many authors writing separate chapters on different aspects of the same topic, sometimes with differing perspectives.
A monograph presents a research topic. It is written by either one author or a small group of authors.


In [ ]:
##

In [11]:
# Feel free to modify filters

PASSING_THRESHOLD = 2.5  # Scores >= this value are passing

filters = (
    (abs(df["score"] - df["new_score"]) > 1.0)  # scores differ by more than 1.0
    | ((df["score"] > PASSING_THRESHOLD) & (df["new_score"] < PASSING_THRESHOLD))  # scores on different sides of threshold
    | ((df["score"] < PASSING_THRESHOLD) & (df["new_score"] > PASSING_THRESHOLD))  # scores on different sides of threshold
) 

# For example, this will select scores that are within TOLERANCE of TARGET_SCORE
TARGET_SCORE = 1.8
TOLERANCE = 0.2
filters = (df["new_score"] - TARGET_SCORE).abs() < TOLERANCE

# Apply filters
df_filtered = df[filters]

df_filtered

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing,mpnet_score,new_score
270,examiner-s-manual-standardization,requirements-for-student-participants,Requirements-for-Student-Participants-6183,The purpose of the CELF-6 Standardization is t...,What is the purpose of the CELF-6 Standardizat...,The purpose of the CELF-6 Standardization is t...,To not have a diagnosis of color blindness,4,3.516423,Excellent job! Your response demonstrates deep...,True,0.936375,1.791737
284,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,Discussion-4857,This study introduces and helps validate TAALE...,What is the purpose of TAALES 2.0 as described...,TAALES 2.0 is a tool for measuring lexical sop...,It is an easy to use freely available versatil...,2,1.906836,"You are on the right track, but your response ...",False,0.846578,1.679042
286,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,A number of suffixes and important features ar...,3,2.899514,Good job. You included some relevant details f...,True,0.793546,1.880504
311,natural-language-processing,text-coherence-and-judgments-of-essay-quality-...,Discussion-II-5867,Multiple Regression Training Set\n\nA linear r...,What percentage of the variance in human evalu...,80% of the variance.,81% of the variance,2,2.067881,"You are on the right track, but your response ...",False,0.624085,1.744666
322,natural-language-processing,sentiment-analysis-and-social-cognition-engine...,Results-5769,Movie review corpus\n\nLIWC indices\n\nMANOVA\...,What was the accuracy rate of the DFA using th...,The accuracy rate was 85.0%.,84.2%-85% accuracy,2,2.004777,"You are on the right track, but your response ...",False,0.581577,1.916633
334,research-methods-in-psychology,5-experimental-and-clinical-psychologists,Clinical-Psychologists-1466t,Psychology is the scientific study of behavior...,What are empirically supported treatments in p...,Treatments that have been shown to work throug...,Treatements that have been shown to work are c...,3,2.771466,Good job. You included some relevant details f...,True,0.931257,1.978881
369,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,The transformation-based tagger constructed ru...,4,3.336455,Excellent job! Your response demonstrates deep...,True,0.855857,1.943286
371,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,A statistical approach was used in the transfo...,4,3.251928,Excellent job! Your response demonstrates deep...,True,0.735563,1.771421
377,mars-petcare,mars-petcare-i,Experiment-Summary-5912,We are planning to cook 126* unique combinatio...,What are the criteria for ranking the top comb...,The criteria include visual quality based on r...,top three solutions for visual representations...,2,1.946577,"You are on the right track, but your response ...",False,0.783254,1.716063
385,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,It assigns the most likely tag based on the tr...,4,3.240489,Excellent job! Your response demonstrates deep...,True,0.936109,1.778334


In [12]:
current_idx = 0

def create_display(df, idx):
    """Create the display for a given row index"""
    if idx < 0 or idx >= len(df):
        return None

    row = df.iloc[idx]

    # Round scores to 2 decimal places
    score = round(row["score"], 2)
    new_score = round(row["new_score"], 2)

    # Determine passing status and styling
    score_passing = score >= PASSING_THRESHOLD
    new_score_passing = new_score >= PASSING_THRESHOLD

    score_bg = "#c8e6c9" if score_passing else "#ffcdd2"
    new_score_bg = "#c8e6c9" if new_score_passing else "#ffcdd2"

    html_content = f"""
    <div style="font-family: Arial, sans-serif; line-height: 1.6;">
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin-bottom: 15px;">
            <strong>Row {idx + 1} of {len(df)}</strong>
        </div>
        
        <div style="background-color: #e8f4f8; padding: 12px; border-left: 4px solid #2196F3; margin-bottom: 15px;">
            <strong>Question:</strong><br>
            {row["question"]}
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <div style="background-color: {score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">Original Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{score}</div>
            </div>
            
            <div style="background-color: {new_score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">New Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{new_score}</div>
            </div>
        </div>
                
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="background-color: #e8f5e9; padding: 15px; border: 2px solid #4CAF50; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #2e7d32;">Reference Answer</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["reference"]}
                </div>
            </div>
            
            <div style="background-color: #fce4ec; padding: 15px; border: 2px solid #E91E63; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #880e4f;">Student Response</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["candidate"]}
                </div>
            </div>
        </div>

        <hr style="margin: 20px 0;">

        <div style="background-color: #fff3e0; padding: 12px; border-left: 4px solid #FF9800; margin-bottom: 15px;">
            <strong>Text/Context:</strong><br>
            {row["text"]}
        </div>

    </div>
    """

    return html_content


# Create output widget
output = widgets.Output()

# Create buttons
prev_button = widgets.Button(description="← Previous", button_style="info")
next_button = widgets.Button(description="Next →", button_style="info")
status_label = widgets.Label(value=f"Row 1 of {len(df_filtered)}")


def on_prev_clicked(b):
    global current_idx
    current_idx = max(0, current_idx - 1)
    update_display()


def on_next_clicked(b):
    global current_idx
    current_idx = min(len(df_filtered) - 1, current_idx + 1)
    update_display()


def update_display():
    output.clear_output(wait=True)
    html_content = create_display(df_filtered, current_idx)
    status_label.value = f"Row {current_idx + 1} of {len(df_filtered)}"
    with output:
        display(HTML(html_content))


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Create layout
button_box = widgets.HBox([prev_button, status_label, next_button])
layout = widgets.VBox([button_box, output])

# Initial display
update_display()

# Show the interface
display(layout)

LH - I am thinking the following for thresholds:
1. (-inf, 1.5)
2. [1.5, 2.1)
3. [2.1, 3.3)
4. [3.3, +inf)